# 3. 모델링

다양한 분류 알고리즘을 적용하고 성능을 비교합니다.
- 기본 모델: RandomForest, LightGBM, XGBoost
- 하이퍼파라미터 튜닝: Optuna
- AutoML: AutoGluon (Stacking + Bagging)
- 예측: AutoGluon Best Model

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

train_data = pd.read_csv('../data/train_processed.csv')
test_raw   = pd.read_csv('../data/test_processed.csv')
test_ids   = pd.read_csv('../data/test_ids.csv')['ID']

X = train_data.drop('label', axis=1)
y = train_data['label']
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

Features: 45, Samples: 7000


## 3.1 기본 모델 비교 (Cross Validation)

In [7]:
import lightgbm as lgb
import xgboost as xgb

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'LightGBM':     lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost':      xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=42, eval_metric='logloss'),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    results[name] = scores.mean()
    print(f"{name}: {scores.mean():.5f} (±{scores.std():.5f})")

RandomForest: 0.73343 (±0.00518)
LightGBM: 0.72829 (±0.01035)
XGBoost: 0.72729 (±0.01165)


## 3.2 하이퍼파라미터 튜닝 (Optuna)

LightGBM과 XGBoost에 대해 Optuna 기반 하이퍼파라미터 탐색을 수행합니다.

In [8]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 100),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = lgb.LGBMClassifier(**params, random_state=42, verbose=-1)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        scores.append(accuracy_score(y.iloc[val_idx], model.predict(X.iloc[val_idx])))
    return np.mean(scores)

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

best_lgbm_config = study_lgbm.best_params
print(f"Best LightGBM Accuracy: {study_lgbm.best_value:.5f}")
print(f"Best Config: {best_lgbm_config}")

Best trial: 39. Best value: 0.737143: 100%|██████████| 50/50 [08:33<00:00, 10.27s/it]

Best LightGBM Accuracy: 0.73714
Best Config: {'n_estimators': 327, 'learning_rate': 0.01861686765726617, 'num_leaves': 63, 'max_depth': 8, 'min_child_samples': 27, 'subsample': 0.7986289349719681, 'colsample_bytree': 0.6518886989411588, 'reg_alpha': 0.09346488306830136, 'reg_lambda': 0.7168554167653843}


In [9]:
def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = xgb.XGBClassifier(**params, random_state=42, eval_metric='logloss')
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        scores.append(accuracy_score(y.iloc[val_idx], model.predict(X.iloc[val_idx])))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

best_xgb_config = study_xgb.best_params
print(f"Best XGBoost Accuracy: {study_xgb.best_value:.5f}")
print(f"Best Config: {best_xgb_config}")

Best trial: 33. Best value: 0.739857: 100%|██████████| 50/50 [28:42<00:00, 34.44s/it]

Best XGBoost Accuracy: 0.73986
Best Config: {'n_estimators': 565, 'learning_rate': 0.014994820924168558, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.6373057674902991, 'colsample_bytree': 0.8513687816443544, 'reg_alpha': 0.0010271617402183052, 'reg_lambda': 0.010325821095077044}


## 3.3 AutoGluon (AutoML)

AutoGluon `best_quality` 프리셋으로 자동 스태킹 + 배깅을 적용합니다.

In [10]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(
    label='label',
    eval_metric='accuracy',
    path='../outputs/ag_models',
    problem_type='binary'
).fit(
    train_data,
    presets='best_quality',
    num_bag_folds=10,
    time_limit=600,
    num_gpus=0,
    dynamic_stacking=False,
    excluded_model_types=['FASTAI', 'NN_TORCH'],
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.6
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          4
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       1.08 GB / 7.86 GB (13.7%)
Disk Space Avail:   320.16 GB / 476.44 GB (67.2%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=10, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "c:\Users\USER\Desktop\Beginner-track\프로젝트\smoking-status-prediction\outputs\ag_models"
Train Data Rows:    7000
Train Data Columns: 45
Label Column:       label
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data .

## 3.4 모델 리더보드 확인 및 Best Model 예측

In [11]:
lb = predictor.leaderboard(train_data, silent=True)
lb_sorted = lb.sort_values(by='score_val', ascending=False)
print("Top 10 모델:")
print(lb_sorted[['model', 'score_val']].head(10).to_string(index=False))

Top 10 모델:
                   model  score_val
     WeightedEnsemble_L2   0.744571
 RandomForestGini_BAG_L1   0.737143
   ExtraTrees_r42_BAG_L1   0.734429
RandomForest_r195_BAG_L1   0.732286
   ExtraTreesEntr_BAG_L1   0.732286
 RandomForestEntr_BAG_L1   0.732000
   ExtraTreesGini_BAG_L1   0.729143


In [13]:
best_model = predictor.model_best
print(f"Best Model: {best_model}")
print(f"Best Val Score: {lb_sorted.iloc[0]['score_val']:.5f}")

pred_proba = predictor.predict_proba(test_raw, model=best_model).iloc[:, 1]

submission = pd.DataFrame({
    'ID': test_ids,
    'label': (pred_proba >= 0.5).astype(int)
})
submission.to_csv('../outputs/submission.csv', index=False)
print(f"제출 파일 저장 완료: {submission.shape}")
submission.head()

Best Model: WeightedEnsemble_L2
Best Val Score: 0.74457
제출 파일 저장 완료: (3000, 2)


,ID,label
0,TEST_0000,0
1,TEST_0001,0
2,TEST_0002,1
3,TEST_0003,1
4,TEST_0004,0
